<a href="https://colab.research.google.com/github/ndvphuc07/HW/blob/main/BaiTapNhom.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [35]:
!pip install gradio folium requests

import gradio as gr
import folium
import random
import requests
from datetime import datetime

# =========================
# CACHE + HISTORY
# =========================
geo_cache = {}
search_history = []

# =========================
# DATA
# =========================
POPULAR_PLACES = [
    "Chợ Bến Thành, Quận 1",
    "Landmark 81, Bình Thạnh",
    "Sân bay Tân Sơn Nhất",
    "Phố đi bộ Nguyễn Huệ",
    "UEH Cơ sở A (Nguyễn Đình Chiểu)",
    "UEH Cơ sở B (Nguyễn Tri Phương)",
    "UEH Cơ sở N (Nguyễn Văn Linh)",
    "UEH Cơ sở V (Nguyễn Kiệm)"
]

FIXED_LOCATIONS = {
    "ueh cơ sở a": (10.7816, 106.6922),
    "nguyễn đình chiểu": (10.7816, 106.6922),
    "ueh cơ sở b": (10.7624, 106.6663),
    "nguyễn tri phương": (10.7624, 106.6663),
    "ueh cơ sở n": (10.7288, 106.7042),
    "nguyễn văn linh": (10.7288, 106.7042),
    "ueh cơ sở v": (10.8030, 106.6800),
    "nguyễn kiệm": (10.8030, 106.6800),
    "bến thành": (10.7726, 106.6980),
    "landmark": (10.7946, 106.7223),
    "sân bay": (10.8149, 106.6634),
}

HEADERS = {"User-Agent": "HopOnApp/1.0"}

# =========================
# AUTOCOMPLETE
# =========================
def get_initial_suggestions():
    return list(dict.fromkeys(search_history + POPULAR_PLACES))

def suggest_address(text):
    if not text or len(text) < 2:
        return gr.update(choices=get_initial_suggestions())

    text_lower = text.lower()
    history_match = [h for h in search_history if text_lower in h.lower()]
    popular_match = [p for p in POPULAR_PLACES if text_lower in p.lower()]

    try:
        url = "https://nominatim.openstreetmap.org/search"
        params = {"q": text, "format": "json", "limit": 5, "countrycodes": "vn"}
        res = requests.get(url, params=params, headers=HEADERS, timeout=3).json()
        api_results = [r["display_name"] for r in res]
    except:
        api_results = []

    return gr.update(choices=list(dict.fromkeys(history_match + popular_match + api_results)))

# =========================
# GEOCODE
# =========================
def get_coords(addr):
    addr_clean = addr.lower().strip()

    for key, coord in FIXED_LOCATIONS.items():
        if key in addr_clean:
            return coord

    if addr in geo_cache:
        return geo_cache[addr]

    try:
        url = "https://nominatim.openstreetmap.org/search"
        params = {"q": addr, "format": "json", "limit": 1, "countrycodes": "vn"}
        res = requests.get(url, params=params, headers=HEADERS, timeout=5).json()

        if not res:
            return None

        lat = float(res[0]["lat"])
        lon = float(res[0]["lon"])
        geo_cache[addr] = (lat, lon)
        return lat, lon
    except:
        return None

# =========================
# ROUTE
# =========================
def get_route(a, b):
    try:
        url = f"http://router.project-osrm.org/route/v1/driving/{a[1]},{a[0]};{b[1]},{b[0]}?overview=full&geometries=geojson"
        data = requests.get(url, timeout=5).json()
        coords = data['routes'][0]['geometry']['coordinates']
        dist = data['routes'][0]['distance'] / 1000
        return [[c[1], c[0]] for c in coords], dist
    except:
        return [a, b], random.uniform(2, 6)

# =========================
# ETA
# =========================
def estimate_time(distance, traffic):
    speed = 18 if traffic >= 8 else 28 if traffic >= 5 else 40
    t = (distance / speed) * 60 + random.uniform(2, 5)
    return max(1, int(t - 2)), max(2, int(t + 3))

# =========================
# FUZZY
# =========================
def calculate_advanced_fuzzy_surge(t, w):
    def trimf(x, a, b, c):
        if x <= a or x >= c: return 0
        if a < x <= b: return (x-a)/(b-a)
        if b < x < c: return (c-x)/(c-b)
        return 0

    r1 = min(trimf(t,0,0,5), trimf(w,0,0,4))
    r2 = max(min(trimf(t,2,5,8), trimf(w,0,0,4)),
             min(trimf(t,0,0,5), trimf(w,2,5,8)))
    r3 = max(trimf(t,6,10,10), trimf(w,6,10,10))

    num = r1*1.0 + r2*1.3 + r3*1.6
    den = r1 + r2 + r3
    return round((num/den) if den else 1.0, 2)

# =========================
# WEATHER
# =========================
def get_weather():
    try:
        url = "https://api.open-meteo.com/v1/forecast?latitude=10.76&longitude=106.66&current_weather=true"
        code = requests.get(url, timeout=2).json()['current_weather']['weathercode']
        if code <= 3: return 1, "☀️ Nắng"
        elif code <= 65: return 7, "🌧️ Mưa"
        else: return 9, "⛈️ Bão"
    except:
        return 2, "🌤️ Bình thường"

# =========================
# MAIN
# =========================
def process(o, d, vehicle, options):
    if not o or not d:
        return "⚠️ Nhập địa điểm!", ""

    A = get_coords(o)
    B = get_coords(d)
    if not A or not B:
        return "❌ Không tìm thấy địa điểm", ""

    hour = datetime.now().hour
    traffic = 8 if 7<=hour<=9 or 17<=hour<=19 else 4

    route, dist = get_route(A, B)
    eta_low, eta_high = estimate_time(dist, traffic)

    weather, w_text = get_weather()
    surge = calculate_advanced_fuzzy_surge(traffic, weather)

    base = 11000 if "Moto" in vehicle else 20000
    per_km = 4500 if "Moto" in vehicle else 10000

    extra = 0
    if options:
        if "📦 Giao hàng nhanh" in options: extra += 5000
        if "🚀 Ưu tiên tài xế 5⭐" in options: extra += 3000

    cost = int((base + dist*per_km)*surge + extra)
    cost = int(cost/1000)*1000

    for place in [o, d]:
        if place not in search_history:
            search_history.insert(0, place)
    search_history[:] = search_history[:10]

    m = folium.Map(location=A, zoom_start=13)
    folium.Marker(A).add_to(m)
    folium.Marker(B).add_to(m)
    folium.PolyLine(route, color="#ff7eb3").add_to(m)

    options_text = "<br>".join(options) if options else "Không có"

    html = f"""
    <div class="info-box">
        <h1>{cost:,} VNĐ</h1>

        🚦 <b>Giao thông:</b> {'Cao điểm' if traffic>6 else 'Ổn định'}<br>
        ⛅ <b>Thời tiết:</b> {w_text}<br>

        ⏱️ <b>Thời gian:</b> {eta_low} - {eta_high} phút<br>
        📏 <b>Khoảng cách:</b> {round(dist,1)} km<br>

        ⚡ <b>Surge:</b> {surge}x

        <hr>

        🧩 <b>Tiện ích:</b><br>
        {options_text}
    </div>
    """

    return html, m._repr_html_()

# =========================
# CSS (CHỈ SỬA PHẦN NÀY)
# =========================
css = """
.gradio-container {
    background: linear-gradient(135deg, #ff9ecf, #ffc1e3, #ffd6ec) !important;
}

body {
    background: #ffc1e3 !important;
}

.card {
    background: rgba(255,255,255,0.9);
    padding:20px;
    border-radius:20px;
}

/* FIX MÀU TRONG BOX */
.info-box {
    background: rgba(255,255,255,0.95);
    backdrop-filter: blur(12px);
    padding: 20px;
    border-radius: 20px;
    text-align: center;
    color: #ff6fa5 !important;
}

.info-box * {
    color: #ff6fa5 !important;
}

.info-box h1 {
    color: #ff4fa3 !important;
}
"""

# =========================
# UI
# =========================
with gr.Blocks(css=css) as app:

    gr.HTML("<h1 style='text-align:center;color:#ff4fa3'>🚗 HOP ON</h1>")

    with gr.Row():
        with gr.Column():
            with gr.Group(elem_classes="card"):

                origin = gr.Dropdown([], allow_custom_value=True, label="Điểm đón")
                dest = gr.Dropdown([], allow_custom_value=True, label="Điểm đến")

                origin.input(suggest_address, origin, origin)
                dest.input(suggest_address, dest, dest)

                origin.focus(lambda: gr.update(choices=get_initial_suggestions()), None, origin)
                dest.focus(lambda: gr.update(choices=get_initial_suggestions()), None, dest)

                vehicle = gr.Radio(["Moto 🏍️","Car 🚗"], value="Moto 🏍️")

                options = gr.CheckboxGroup([
                    "🤫 Chuyến xe im lặng",
                    "📦 Giao hàng nhanh",
                    "🚀 Ưu tiên tài xế 5⭐"
                ])

                btn = gr.Button("🚀 TÌM XE")

        with gr.Column():
            price = gr.HTML()
            map_out = gr.HTML()

    btn.click(process, [origin, dest, vehicle, options], [price, map_out])

app.launch(share=True, inline=False)

/tmp/ipykernel_3368/2566675870.py:255: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=css) as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1393b489f450684ec2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
